# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [116]:
# read transitions (columns are action, state, next_state, probability)
transitions = []
with open("twizzleflu-transitions.tsv") as f:
    next(f)
    for line in f:
        action, state, next_state, prob = line.strip().split("\t")
        transitions.append((action, state, next_state, float(prob)))

# read rewards (columns are action, state, reward)
rewards = []
with open("twizzleflu-rewards.tsv") as f:
    next(f)
    for line in f:
        action, state, reward = line.strip().split("\t")
        rewards.append((action, state, -float(reward)))  # flip sign for discomfort

# collect all states
states_set = set()
for (a, s, ns, p) in transitions:
    states_set.add(s)
    states_set.add(ns)

states = sorted(list(states_set))  # sorted so the file order is stable

# start values at 0
values = {}
for s in states:
    values[s] = 0.0

# do-nothing evaluation by repeating updates
for i in range(10000):
    new_values = values.copy()
    biggest_change = 0.0

    for s in states:
        if s == "recovered":
            new_values[s] = 0.0
        else:
            # reward for do-nothing at this state
            r = 0.0
            for (a, st, rew) in rewards:
                if a == "do-nothing" and st == s:
                    r = rew
                    break

            # expected next value under do-nothing
            total = 0.0
            for (a, st, ns, p) in transitions:
                if a == "do-nothing" and st == s:
                    total += p * values[ns]

            new_values[s] = r + total
            change = abs(new_values[s] - values[s])
            if change > biggest_change:
                biggest_change = change

    values = new_values
    if biggest_change < 1e-12:
        break


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [117]:
# save tsv
with open("do-nothing-discomfort.tsv", "w") as f:
    f.write("state\texpected_discomfort\n")
    for s in states:
        f.write(s + "\t" + str(values[s]) + "\n")

print("saved do-nothing-discomfort.tsv")

saved do-nothing-discomfort.tsv


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [118]:
# read transitions
transitions = []
with open("twizzleflu-transitions.tsv") as f:
    next(f)
    for line in f:
        action, state, next_state, prob = line.strip().split("\t")
        transitions.append((action, state, next_state, float(prob)))

# states + actions
states_set = set()
actions_set = set()
for (a, s, ns, p) in transitions:
    actions_set.add(a)
    states_set.add(s)
    states_set.add(ns)

states = sorted(list(states_set))
actions = sorted(list(actions_set))

# duration reward: -1 if symptoms, else 0
def duration_reward(s):
    if str(s).startswith("symptoms-"):
        return -1.0
    return 0.0

# expected next value
def expected_next_value(a, s, values):
    total = 0.0
    for (aa, ss, ns, p) in transitions:
        if aa == a and ss == s:
            total += p * values[ns]
    return total

# value iteration (gamma=1)
values = {}
for s in states:
    values[s] = 0.0
values["recovered"] = 0.0

for i in range(10000):
    new_values = values.copy()
    biggest_change = 0.0

    for s in states:
        if s == "recovered":
            new_values[s] = 0.0
        else:
            best = None
            for a in actions:
                q = duration_reward(s) + expected_next_value(a, s, values)
                if (best is None) or (q > best):
                    best = q
            new_values[s] = best

        change = abs(new_values[s] - values[s])
        if change > biggest_change:
            biggest_change = change

    values = new_values
    if biggest_change < 1e-12:
        break

# greedy policy
minimum_duration_actions = {}
for s in states:
    if s == "recovered":
        minimum_duration_actions[s] = "do-nothing"
    else:
        best_a = None
        best = None
        for a in actions:
            q = duration_reward(s) + expected_next_value(a, s, values)
            if (best is None) or (q > best):
                best = q
                best_a = a
        minimum_duration_actions[s] = best_a

Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [119]:
with open("minimum-discomfort-actions.tsv", "w") as f:
    f.write("state\taction\n")
    for s in states:
        f.write(s + "\t" + minimum_discomfort_actions[s] + "\n")

print("saved minimum-discomfort-actions.tsv")

saved minimum-discomfort-actions.tsv


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [120]:
# read transitions (action, state, next_state, probability)
transitions = []
with open("twizzleflu-transitions.tsv") as f:
    next(f)
    for line in f:
        action, state, next_state, prob = line.strip().split("\t")
        transitions.append((action, state, next_state, float(prob)))

# read rewards (action, state, reward) and flip sign to get discomfort
rewards = []
with open("twizzleflu-rewards.tsv") as f:
    next(f)
    for line in f:
        action, state, reward = line.strip().split("\t")
        rewards.append((action, state, -float(reward)))

# read the policy from Q2 output
policy = {}
with open("minimum-discomfort-actions.tsv") as f:
    next(f)
    for line in f:
        state, action = line.strip().split("\t")
        policy[state] = action

# collect all states
states_set = set()
for (a, s, ns, p) in transitions:
    states_set.add(s)
    states_set.add(ns)
states = sorted(list(states_set))

# helper: reward for (action, state)
def get_reward(a, s):
    for (aa, ss, rr) in rewards:
        if aa == a and ss == s:
            return rr
    return 0.0

# expected next value for a fixed (action, state)
def expected_next_value(a, s, values):
    total = 0.0
    for (aa, ss, ns, p) in transitions:
        if aa == a and ss == s:
            total += p * values[ns]
    return total

# policy evaluation by 반복 (gamma=1)
values = {}
for s in states:
    values[s] = 0.0
values["recovered"] = 0.0

for i in range(10000):
    new_values = values.copy()
    biggest_change = 0.0

    for s in states:
        if s == "recovered":
            new_values[s] = 0.0
        else:
            a = policy[s]
            r = get_reward(a, s)
            nxt = expected_next_value(a, s, values)
            new_values[s] = r + nxt

        change = abs(new_values[s] - values[s])
        if change > biggest_change:
            biggest_change = change

    values = new_values
    if biggest_change < 1e-12:
        break

minimum_discomfort_values = values

Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [121]:
with open("minimum-discomfort-values.tsv", "w") as f:
    f.write("state\texpected_discomfort\n")
    for s in states:
        f.write(s + "\t" + str(minimum_discomfort_values[s]) + "\n")

print("saved minimum-discomfort-values.tsv")

saved minimum-discomfort-values.tsv


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [122]:
# read transitions so we can get the list of states + actions
transitions = []
with open("twizzleflu-transitions.tsv") as f:
    next(f)
    for line in f:
        action, state, next_state, prob = line.strip().split("\t")
        transitions.append((action, state, next_state, float(prob)))

states_set = set()
actions_set = set()
for (a, s, ns, p) in transitions:
    actions_set.add(a)
    states_set.add(s)
    states_set.add(ns)

states = sorted(list(states_set))
actions = sorted(list(actions_set))

# sick = symptoms-* states
def is_sick(s):
    return str(s).startswith("symptoms-")

# make duration rewards for every (action, state)
duration_rows = []
for a in actions:
    for s in states:
        if is_sick(s):
            r = -1.0
        else:
            r = 0.0
        duration_rows.append((a, s, r))

Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [123]:
with open("duration-rewards.tsv", "w") as f:
    f.write("action\tstate\treward\n")
    for (a, s, r) in duration_rows:
        f.write(a + "\t" + s + "\t" + str(r) + "\n")

print("saved duration-rewards.tsv")

saved duration-rewards.tsv


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [124]:
# read transitions
transitions = []
with open("twizzleflu-transitions.tsv") as f:
    next(f)
    for line in f:
        action, state, next_state, prob = line.strip().split("\t")
        transitions.append((action, state, next_state, float(prob)))

# read duration rewards
duration_rewards = []
with open("duration-rewards.tsv") as f:
    next(f)
    for line in f:
        action, state, reward = line.strip().split("\t")
        duration_rewards.append((action, state, float(reward)))

# states + actions
states_set = set()
actions_set = set()
for (a, s, ns, p) in transitions:
    actions_set.add(a)
    states_set.add(s)
    states_set.add(ns)

states = sorted(list(states_set))
actions = sorted(list(actions_set))

# reward lookup
def get_reward(a, s):
    for (aa, ss, rr) in duration_rewards:
        if aa == a and ss == s:
            return rr
    return 0.0

# expected next value
def expected_next_value(a, s, values):
    total = 0.0
    for (aa, ss, ns, p) in transitions:
        if aa == a and ss == s:
            total += p * values[ns]
    return total

# value iteration (gamma=1)
values = {}
for s in states:
    values[s] = 0.0
values["recovered"] = 0.0

for i in range(10000):
    new_values = values.copy()
    biggest_change = 0.0

    for s in states:
        if s == "recovered":
            new_values[s] = 0.0
        else:
            best = None
            for a in actions:
                r = get_reward(a, s)
                nxt = expected_next_value(a, s, values)
                q = r + nxt
                if (best is None) or (q > best):
                    best = q
            new_values[s] = best

        change = abs(new_values[s] - values[s])
        if change > biggest_change:
            biggest_change = change

    values = new_values
    if biggest_change < 1e-12:
        break

# greedy policy from final values
policy = {}
for s in states:
    if s == "recovered":
        policy[s] = "do-nothing"
    else:
        best_a = None
        best = None
        for a in actions:
            r = get_reward(a, s)
            nxt = expected_next_value(a, s, values)
            q = r + nxt
            if (best is None) or (q > best):
                best = q
                best_a = a
        policy[s] = best_a

minimum_duration_actions = policy

Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [125]:
with open("minimum-duration-actions.tsv", "w") as f:
    f.write("state\taction\n")
    for s in states:
        f.write(s + "\t" + minimum_duration_actions[s] + "\n")

print("saved minimum-duration-actions.tsv")

saved minimum-duration-actions.tsv


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [126]:
# read transitions
transitions = []
with open("twizzleflu-transitions.tsv") as f:
    next(f)
    for line in f:
        action, state, next_state, prob = line.strip().split("\t")
        transitions.append((action, state, next_state, float(prob)))

# read duration rewards
duration_rewards = []
with open("duration-rewards.tsv") as f:
    next(f)
    for line in f:
        action, state, reward = line.strip().split("\t")
        duration_rewards.append((action, state, float(reward)))

# read the minimum-duration policy from Q5 output
policy = {}
with open("minimum-duration-actions.tsv") as f:
    next(f)
    for line in f:
        state, action = line.strip().split("\t")
        policy[state] = action

# collect states
states_set = set()
for (a, s, ns, p) in transitions:
    states_set.add(s)
    states_set.add(ns)
states = sorted(list(states_set))

# reward lookup
def get_reward(a, s):
    for (aa, ss, rr) in duration_rewards:
        if aa == a and ss == s:
            return rr
    return 0.0

# expected next value
def expected_next_value(a, s, values):
    total = 0.0
    for (aa, ss, ns, p) in transitions:
        if aa == a and ss == s:
            total += p * values[ns]
    return total

# policy evaluation (gamma=1)
values = {}
for s in states:
    values[s] = 0.0
values["recovered"] = 0.0

for i in range(10000):
    new_values = values.copy()
    biggest_change = 0.0

    for s in states:
        if s == "recovered":
            new_values[s] = 0.0
        else:
            a = policy[s]
            r = get_reward(a, s)
            nxt = expected_next_value(a, s, values)
            new_values[s] = r + nxt

        change = abs(new_values[s] - values[s])
        if change > biggest_change:
            biggest_change = change

    values = new_values
    if biggest_change < 1e-12:
        break

# expected sick days = - expected reward (since reward is -1 per sick day)
expected_sick_days = {}
for s in states:
    expected_sick_days[s] = -values[s]

minimum_duration_days = expected_sick_days

Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [127]:
with open("minimum-duration-days.tsv", "w") as f:
    f.write("state\texpected_sick_days\n")
    for s in states:
        f.write(s + "\t" + str(minimum_duration_days[s]) + "\n")

print("saved minimum-duration-days.tsv")

saved minimum-duration-days.tsv


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [128]:
# read transitions
transitions = []
with open("twizzleflu-transitions.tsv") as f:
    next(f)
    for line in f:
        action, state, next_state, prob = line.strip().split("\t")
        transitions.append((action, state, next_state, float(prob)))

# read rewards and flip sign to get discomfort
discomfort_rewards = []
with open("twizzleflu-rewards.tsv") as f:
    next(f)
    for line in f:
        action, state, reward = line.strip().split("\t")
        discomfort_rewards.append((action, state, -float(reward)))

# read both policies
policy_speed = {}
with open("minimum-duration-actions.tsv") as f:
    next(f)
    for line in f:
        state, action = line.strip().split("\t")
        policy_speed[state] = action

policy_pamper = {}
with open("minimum-discomfort-actions.tsv") as f:
    next(f)
    for line in f:
        state, action = line.strip().split("\t")
        policy_pamper[state] = action

# collect states
states_set = set()
for (a, s, ns, p) in transitions:
    states_set.add(s)
    states_set.add(ns)
states = sorted(list(states_set))

# discomfort reward lookup
def get_discomfort(a, s):
    for (aa, ss, rr) in discomfort_rewards:
        if aa == a and ss == s:
            return rr
    return 0.0

# expected next value for a fixed (action, state)
def expected_next_value(a, s, values):
    total = 0.0
    for (aa, ss, ns, p) in transitions:
        if aa == a and ss == s:
            total += p * values[ns]
    return total

# evaluate a policy (returns expected discomfort values)
def eval_policy(policy):
    values = {}
    for s in states:
        values[s] = 0.0
    values["recovered"] = 0.0

    for i in range(10000):
        new_values = values.copy()
        biggest_change = 0.0

        for s in states:
            if s == "recovered":
                new_values[s] = 0.0
            else:
                a = policy[s]
                r = get_discomfort(a, s)
                nxt = expected_next_value(a, s, values)
                new_values[s] = r + nxt

            change = abs(new_values[s] - values[s])
            if change > biggest_change:
                biggest_change = change

        values = new_values
        if biggest_change < 1e-12:
            break

    return values

speed_discomfort_values = eval_policy(policy_speed)
pamper_discomfort_values = eval_policy(policy_pamper)

comparison_rows = []
for s in states:
    comparison_rows.append((s, speed_discomfort_values[s], pamper_discomfort_values[s]))

Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [129]:
with open("policy-comparison.tsv", "w") as f:
    f.write("state\tspeed_discomfort\tminimize_discomfort\n")
    for (s, sd, md) in comparison_rows:
        f.write(s + "\t" + str(sd) + "\t" + str(md) + "\n")

print("saved policy-comparison.tsv")

saved policy-comparison.tsv


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.